We can use our MI300x to fine-tune a local LLM. 

This approach is cost-effective and helps maintain confidentiality.

OpenAI has released a fine-tuning guideline for gpt-oss: https://cookbook.openai.com/articles/gpt-oss/fine-tune-transfomers#inference

The code here shows an example using gpt-oss:20b on an AMD server.

Currently, we are running gpt-oss:20b in zero-shot mode.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json

In [2]:
kirk_labeled = pd.read_excel('text/kirk_labeled_zeroshot.xlsx')

In [3]:
kirk_labeled.head()

,Message,Sentiment
0,Guess the universe finally muted CK’s podcast.,mocking
1,The bar is quieter—and happier—without CK.,mocking
2,May CK’s soul find peace in eternity.,mourning
3,The jukebox broke every time Charlie Kirk pick...,mocking
4,"Kirk’s book closed, but was it ever worth read...",mocking


In [4]:
import ollama

In [5]:
kirk_labeled['Message'].iloc[0]


'Guess the universe finally muted CK’s podcast.'

In [6]:
response = ollama.chat(
        model="gpt-oss:20b",
        messages=[
            {"role": "system", "content": "You are a professional text classifier. I will give you a message, and you will tell me whether the message is mocking or mourning. Please respond with only mocking or mourning in lowercase. Do not include any other information, the original text, or explanations."},
            {"role": "user", "content": "The message is here:" + kirk_labeled['Message'].iloc[0]}
        ]
    )

In [7]:
response['message']['content']

'mocking'

In [8]:
result = []

for i in range(len(kirk_labeled)):
    response = ollama.chat(
        model="gpt-oss:20b",
        messages=[
            {"role": "system", "content": "You are a professional text classifier. I will give you a message, and you will tell me whether the message is mocking or mourning. Please respond with only mocking or mourning in lowercase. Do not include any other information, the original text, or explanations."},
            {"role": "user", "content": "The message is here:" + kirk_labeled['Message'].iloc[i]}
        ]
    )
    
    
    print(response['message']['content'])
    result.append(response['message']['content'])  

mocking
mocking
mourning
mocking
mocking
mourning
mourning
mourning
mocking
mocking
mocking
mocking
mourning
mocking
mocking
mocking
mourning
mocking
mocking
mocking
mourning
mocking
mocking
mocking
mocking
mocking
mocking
mourning
mourning
mocking
mourning
mourning
mocking
mocking
mocking
mocking
mourning
mocking
mocking
mourning
mocking
mocking
mourning
mocking
mocking
mocking
mocking
mourning
mocking
mocking
mourning
mourning
mourning
mocking
mocking
mourning
mourning
mocking
mourning
mocking
mocking
mocking
mocking
mocking
mocking
mourning
mocking
mocking
mourning
mocking
mourning
mocking
mourning
mourning
mourning
mourning
mourning
mourning
mourning
mourning
mourning
mourning
mourning
mourning
mocking
mocking
mocking
mocking
mocking
mocking
mocking
mocking
mocking
mocking
mourning
mourning
mocking
mocking
mocking
mocking


In [9]:
result

['mocking',
 'mocking',
 'mourning',
 'mocking',
 'mocking',
 'mourning',
 'mourning',
 'mourning',
 'mocking',
 'mocking',
 'mocking',
 'mocking',
 'mourning',
 'mocking',
 'mocking',
 'mocking',
 'mourning',
 'mocking',
 'mocking',
 'mocking',
 'mourning',
 'mocking',
 'mocking',
 'mocking',
 'mocking',
 'mocking',
 'mocking',
 'mourning',
 'mourning',
 'mocking',
 'mourning',
 'mourning',
 'mocking',
 'mocking',
 'mocking',
 'mocking',
 'mourning',
 'mocking',
 'mocking',
 'mourning',
 'mocking',
 'mocking',
 'mourning',
 'mocking',
 'mocking',
 'mocking',
 'mocking',
 'mourning',
 'mocking',
 'mocking',
 'mourning',
 'mourning',
 'mourning',
 'mocking',
 'mocking',
 'mourning',
 'mourning',
 'mocking',
 'mourning',
 'mocking',
 'mocking',
 'mocking',
 'mocking',
 'mocking',
 'mocking',
 'mourning',
 'mocking',
 'mocking',
 'mourning',
 'mocking',
 'mourning',
 'mocking',
 'mourning',
 'mourning',
 'mourning',
 'mourning',
 'mourning',
 'mourning',
 'mourning',
 'mourning',
 'mourni

In [10]:
kirk_labeled['ChatGPT'] = result

In [11]:
kirk_labeled

,Message,Sentiment,ChatGPT
0,Guess the universe finally muted CK’s podcast.,mocking,mocking
1,The bar is quieter—and happier—without CK.,mocking,mocking
2,May CK’s soul find peace in eternity.,mourning,mourning
3,The jukebox broke every time Charlie Kirk pick...,mocking,mocking
4,"Kirk’s book closed, but was it ever worth read...",mocking,mocking
...,...,...,...
95,"Thank you for all the memories, CK.",mourning,mourning
96,Guess Charlie Kirk couldn’t handle life anymore.,mocking,mocking
97,Even the credits rolled faster just to end Cha...,mocking,mocking
98,"Like a jukebox eating your coin, CK was always...",mocking,mocking


In [12]:
kirk_labeled['compare'] = np.where(kirk_labeled['Sentiment'] == kirk_labeled['ChatGPT'], 1, 0)

In [13]:
kirk_labeled

,Message,Sentiment,ChatGPT,compare
0,Guess the universe finally muted CK’s podcast.,mocking,mocking,1
1,The bar is quieter—and happier—without CK.,mocking,mocking,1
2,May CK’s soul find peace in eternity.,mourning,mourning,1
3,The jukebox broke every time Charlie Kirk pick...,mocking,mocking,1
4,"Kirk’s book closed, but was it ever worth read...",mocking,mocking,1
...,...,...,...,...
95,"Thank you for all the memories, CK.",mourning,mourning,1
96,Guess Charlie Kirk couldn’t handle life anymore.,mocking,mocking,1
97,Even the credits rolled faster just to end Cha...,mocking,mocking,1
98,"Like a jukebox eating your coin, CK was always...",mocking,mocking,1


In [14]:
kirk_labeled['compare'].sum() / 100


np.float64(0.93)

#### Finetuning gpt-oss:20b

Since I’m not sure about the full capacity of the MI300x, I used the QLoRA method, which requires the least resources for fine-tuning.

Note that the data format for fine-tuning gpt-oss:20b with QLoRA is different from the format used with the OpenAI API.

In [15]:
SYSTEM_PROMPT = ("You are a professional text classifier. I will give you a message, and you will tell me whether the message is mocking or mourning. Please respond with only mocking or mourning in lowercase. Do not include any other information, the original text, or explanations.")

In [16]:
def format_example(msg, label):
    return (
        "[SYSTEM]\n"
        + SYSTEM_PROMPT
        + "\n[/SYSTEM]\n"
        + "[USER]\n"
        + "The message is here: "
        + msg
        + "\n[/USER]\n"
        + "[ASSISTANT]\n"
        + label
    )

In [17]:
kirk_labeled["text"] = kirk_labeled.apply(lambda r: format_example(r["Message"], r["Sentiment"]), axis=1)

In [18]:
kirk_labeled

,Message,Sentiment,ChatGPT,compare,text
0,Guess the universe finally muted CK’s podcast.,mocking,mocking,1,[SYSTEM]\nYou are a professional text classifi...
1,The bar is quieter—and happier—without CK.,mocking,mocking,1,[SYSTEM]\nYou are a professional text classifi...
2,May CK’s soul find peace in eternity.,mourning,mourning,1,[SYSTEM]\nYou are a professional text classifi...
3,The jukebox broke every time Charlie Kirk pick...,mocking,mocking,1,[SYSTEM]\nYou are a professional text classifi...
4,"Kirk’s book closed, but was it ever worth read...",mocking,mocking,1,[SYSTEM]\nYou are a professional text classifi...
...,...,...,...,...,...
95,"Thank you for all the memories, CK.",mourning,mourning,1,[SYSTEM]\nYou are a professional text classifi...
96,Guess Charlie Kirk couldn’t handle life anymore.,mocking,mocking,1,[SYSTEM]\nYou are a professional text classifi...
97,Even the credits rolled faster just to end Cha...,mocking,mocking,1,[SYSTEM]\nYou are a professional text classifi...
98,"Like a jukebox eating your coin, CK was always...",mocking,mocking,1,[SYSTEM]\nYou are a professional text classifi...


In [19]:
import random
seed = 42
random.seed(seed)
indices = list(range(len(kirk_labeled)))
random.shuffle(indices)
val_ratio = 0.05
val_size = max(1, int(len(kirk_labeled) * val_ratio))
val_idx = set(indices[:val_size])

train_df = kirk_labeled.loc[[i for i in range(len(kirk_labeled)) if i not in val_idx]].reset_index(drop=True)
val_df   = kirk_labeled.loc[[i for i in range(len(kirk_labeled)) if i in val_idx]].reset_index(drop=True)

In [20]:
train_df

,Message,Sentiment,ChatGPT,compare,text
0,Guess the universe finally muted CK’s podcast.,mocking,mocking,1,[SYSTEM]\nYou are a professional text classifi...
1,The bar is quieter—and happier—without CK.,mocking,mocking,1,[SYSTEM]\nYou are a professional text classifi...
2,May CK’s soul find peace in eternity.,mourning,mourning,1,[SYSTEM]\nYou are a professional text classifi...
3,The jukebox broke every time Charlie Kirk pick...,mocking,mocking,1,[SYSTEM]\nYou are a professional text classifi...
4,"Kirk’s book closed, but was it ever worth read...",mocking,mocking,1,[SYSTEM]\nYou are a professional text classifi...
...,...,...,...,...,...
90,"Thank you for all the memories, CK.",mourning,mourning,1,[SYSTEM]\nYou are a professional text classifi...
91,Guess Charlie Kirk couldn’t handle life anymore.,mocking,mocking,1,[SYSTEM]\nYou are a professional text classifi...
92,Even the credits rolled faster just to end Cha...,mocking,mocking,1,[SYSTEM]\nYou are a professional text classifi...
93,"Like a jukebox eating your coin, CK was always...",mocking,mocking,1,[SYSTEM]\nYou are a professional text classifi...


In [21]:
from datasets import Dataset, DatasetDict

ds = DatasetDict({
    "train": Dataset.from_pandas(train_df[["text"]], preserve_index=False),
    "validation": Dataset.from_pandas(val_df[["text"]], preserve_index=False)
})

In [22]:
ds

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 95
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 5
    })
})

In [23]:
ds.save_to_disk("train_data")

Saving the dataset (0/1 shards):   0%|          | 0/95 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/5 [00:00<?, ? examples/s]

The parameters and settings for MI300x fine-tuning here are simplified to minimize errors and save time. 
As a result, the performance of the fine-tuned model may be worse than zero-shot.

If your project attempts fine-tuning, you will need to optimize the process. 

We can also consult with engineers from AIA, AMD, and ITRI.

In [24]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("openai/gpt-oss-20b")

/opt/conda/envs/py_3.10/lib/python3.10/site-packages/redis/connection.py:77: UserWarning: redis-py works best with hiredis. Please consider installing
  warnings.warn(msg)


In [ ]:
# import os
# os.environ["TRANSFORMERS_NO_TF"] = "1"       # don't try to import TensorFlow
# os.environ["TRANSFORMERS_NO_FLAX"] = "1"     # (optional) don't try to import Flax/JAX

In [31]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, Mxfp4Config

In [26]:
DATA_DIR = "train_data"
OUTPUT_DIR = "gpt-oss-20b-lora"

In [32]:
model_id = "openai/gpt-oss-20b"

quant_config = Mxfp4Config(dequantize=True)  # use MXFP4 config

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    trust_remote_code=True,
    quantization_config=quant_config,
    dtype=torch.bfloat16,          # or torch.float16 if bf16 fails
    device_map="auto",
    low_cpu_mem_usage=True,
)

bitsandbytes library load error: Configured ROCm binary not found at /opt/conda/envs/py_3.10/lib/python3.10/site-packages/bitsandbytes/libbitsandbytes_rocm64.so
 If you are using Intel CPU/XPU, please install intel_extension_for_pytorch to enable required ops
Traceback (most recent call last):
  File "/opt/conda/envs/py_3.10/lib/python3.10/site-packages/bitsandbytes/cextension.py", line 318, in <module>
    lib = get_native_library()
  File "/opt/conda/envs/py_3.10/lib/python3.10/site-packages/bitsandbytes/cextension.py", line 282, in get_native_library
    raise RuntimeError(f"Configured {BNB_BACKEND} binary not found at {cuda_binary_path}")
RuntimeError: Configured ROCm binary not found at /opt/conda/envs/py_3.10/lib/python3.10/site-packages/bitsandbytes/libbitsandbytes_rocm64.so


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [34]:
from peft import LoraConfig, get_peft_model

In [35]:
lora_config = LoraConfig(
    r=16,                         # rank
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],  # 依模型結構調整
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

In [36]:
model = get_peft_model(model, lora_config)

In [38]:
from datasets import load_from_disk

In [39]:
dataset = load_from_disk(DATA_DIR)

In [41]:
from transformers import TrainingArguments

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=2,                  
    per_device_train_batch_size=1,       
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=16,      
    learning_rate=2e-4,                  
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    logging_steps=20,
    #evaluation_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    bf16=True,
    gradient_checkpointing=True,        
    report_to="none",
)


In [45]:
from trl import SFTTrainer

In [60]:
sft_trainer_kwargs = {
    "model": model,
    "train_dataset": dataset["train"],
    "eval_dataset": dataset.get("validation", None),
    "args": training_args,
    "max_seq_length": 512,
    "packing": True,
    "formatting_func": formatting_func,
}


In [62]:
import inspect

In [ ]:
from trl import SFTTrainer

def formatting_func(example):
    return example["text"]

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    args=training_args,
    formatting_func=formatting_func,
    processing_class=tokenizer,
)

trainer.train()


Applying formatting function to train dataset:   0%|          | 0/95 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/95 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/95 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/95 [00:00<?, ? examples/s]

Applying formatting function to eval dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

...

Tokenizing eval dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 199998}.


Step,Training Loss


TrainOutput(global_step=12, training_loss=2.3792378107706704, metrics={'train_runtime': 492.0942, 'train_samples_per_second': 0.386, 'train_steps_per_second': 0.024, 'total_flos': 2004006005938944.0, 'train_loss': 2.3792378107706704, 'entropy': 2.4401628557004424, 'num_tokens': 16418.0, 'mean_token_accuracy': 0.5528736163126795, 'epoch': 2.0})

In [ ]:
#Once fine-tuning is successful, we can save the model and use it for inference.


trainer.model.save_pretrained("gpt-oss-20b-lora")
tokenizer.save_pretrained("gpt-oss-20b-lora")

('gpt-oss-20b-lora/tokenizer_config.json',
 'gpt-oss-20b-lora/special_tokens_map.json',
 'gpt-oss-20b-lora/chat_template.jinja',
 'gpt-oss-20b-lora/tokenizer.json')

#Inference

In [78]:
from peft import PeftModel

In [ ]:
BASE_MODEL  = "openai/gpt-oss-20b"     
ADAPTER_DIR = "gpt-oss-20b-lora"       

In [80]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)

In [81]:
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

With the LoRA method, the adaptation information is saved in a folder, and we can merge these adapted components into the base model (gpt-oss:20b).

In [ ]:
# === Load the LoRA adapter and merge it into the base_model ===
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model.eval()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): GptOssForCausalLM(
      (model): GptOssModel(
        (embed_tokens): Embedding(201088, 2880, padding_idx=199999)
        (layers): ModuleList(
          (0-23): 24 x GptOssDecoderLayer(
            (self_attn): GptOssAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2880, out_features=4096, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2880, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
            

In [83]:
SYSTEM_PROMPT = (
    "You are a professional text classifier. I will give you a message, "
    "and you will tell me whether the message is mocking or mourning. "
    "Please respond with only mocking or mourning in lowercase. "
    "Do not include any other information, the original text, or explanations."
)

def build_prompt(message: str) -> str:
    return (
        "[SYSTEM]\n" + SYSTEM_PROMPT + "\n[/SYSTEM]\n"
        "[USER]\n" + message.strip() + "\n[/USER]\n"
        "[ASSISTANT]\n"
    )


In [ ]:
def classify(message: str) -> str:
    prompt = build_prompt(message)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=3,       
            do_sample=False,        
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id
        )
    gen_text = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    gen_text = gen_text.strip().split()[0].lower()
    return gen_text


In [86]:
print(classify("Guess the universe finally muted CK’s podcast."))
print(classify("We lost a good soul today, may he rest in peace."))

mocking
mourning


In [87]:
kirk_checkfinetune = pd.read_excel('text/kirk_checkfinetuning.xlsx')

In [89]:
results = []
for i, row in kirk_checkfinetune.iterrows():
    pred = classify(row["Message"])
    results.append(pred)

In [90]:
kirk_checkfinetune["GPTOSS"] = results

In [91]:
kirk_checkfinetune

,Message,Sentiment,GPTOSS
0,Feels like the Wi-Fi finally cut CK off.,mocking,mocking
1,The neighborhood feels like a porch without ro...,mourning,mourning
2,CK’s chapter closed like a library overdue not...,mocking,mourning
3,"Not mourning—celebrating, honestly.",mocking,mocking
4,May we all honor Charlie Kirk by living kindly.,mourning,mourning
...,...,...,...
95,CK was a warm cup of coffee spilled too quickly.,mourning,mourning
96,The laugh track was tired of faking it for CK.,mocking,mocking
97,The lights dimmed on our favorite sitcom chara...,mourning,mourning
98,Not sure why people are crying over Charlie Kirk.,mocking,mocking


In [92]:
kirk_checkfinetune['compare'] = np.where(kirk_checkfinetune['Sentiment'] == kirk_checkfinetune['GPTOSS'], 1, 0)
kirk_checkfinetune

,Message,Sentiment,GPTOSS,compare
0,Feels like the Wi-Fi finally cut CK off.,mocking,mocking,1
1,The neighborhood feels like a porch without ro...,mourning,mourning,1
2,CK’s chapter closed like a library overdue not...,mocking,mourning,0
3,"Not mourning—celebrating, honestly.",mocking,mocking,1
4,May we all honor Charlie Kirk by living kindly.,mourning,mourning,1
...,...,...,...,...
95,CK was a warm cup of coffee spilled too quickly.,mourning,mourning,1
96,The laugh track was tired of faking it for CK.,mocking,mocking,1
97,The lights dimmed on our favorite sitcom chara...,mourning,mourning,1
98,Not sure why people are crying over Charlie Kirk.,mocking,mocking,1


In [93]:
kirk_checkfinetune['compare'].sum() / 100

np.float64(0.76)